In [ ]:
from datetime import datetime
import re
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def create_unique_grain_id(
    grain_filename: str,
    grain_x_coord: float,
    grain_y_coord: float,
) -> str:
    """
    Create a unique grain id based on the filename and the position of the grain, hopefully resistant to changes in the
    dataset.


    Parameters
    ----------
    grain_filename : str
        The filename of the grain.
    grain_x_coord : float
        The x-coordinate of the grain in nanometres.
    grain_y_coord : float
        The y-coordinate of the grain in nanometres.


    Returns
    -------
    str
        A unique grain id.
    """
    # round the coords to the nearest 5 nanometres
    grain_x_coord = round(grain_x_coord / 5e-9) * 5e-9
    grain_y_coord = round(grain_y_coord / 5e-9) * 5e-9
    # deop the nanometre part of the coords
    grain_x_coord = int(grain_x_coord * 1e9)
    grain_y_coord = int(grain_y_coord * 1e9)
    return f"{grain_filename}_{grain_x_coord}_x_{grain_y_coord}_y"

In [ ]:
path_base_old = Path("/Users/sylvi/topo_data/dna_damage_cache/analysis_results_pre_new_controls")
defect_grain_statistics_df = pd.read_csv(path_base_old / "defect_grain_statistics.csv")

mapping: list[tuple[int, str]] = []

# iterate over each row
for index, row in defect_grain_statistics_df.iterrows():
    # get the image name from the "image_x" column
    image_name = row["image_x"] + ".spm"
    grain_id: int = row["grain_id"]
    assert isinstance(grain_id, int), f"grain_id is not an int: {grain_id}"
    x_coord: float = row["centre_x"]
    assert isinstance(x_coord, float), f"x_coord is not a float: {x_coord}"
    y_coord: float = row["centre_y"]
    assert isinstance(y_coord, float), f"y_coord is not a float: {y_coord}"
    unique_id = create_unique_grain_id(image_name, x_coord, y_coord)
    mapping.append((grain_id, unique_id))

print(f"Created mapping for {len(mapping)} grains.")

# save the mapping to a csv file
pd.DataFrame(mapping, columns=["old_grain_id", "unique_id"]).to_csv(
    path_base_old / "old_grain_id_to_unique_id_mapping.csv", index=False
)

In [ ]:
path_base_new = Path("/Users/sylvi/topo_data/dna_damage_cache/analysis_results")
defect_grain_statistics_df = pd.read_csv(path_base_new / "defect_grain_statistics.csv")

mapping: list[tuple[int, str]] = []

# iterate over each row
for index, row in defect_grain_statistics_df.iterrows():
    # get the image name from the "image_x" column
    image_name = row["image_x"] + ".spm"
    grain_id: int = row["grain_id"]
    assert isinstance(grain_id, int), f"grain_id is not an int: {grain_id}"
    x_coord: float = row["centre_x"]
    assert isinstance(x_coord, float), f"x_coord is not a float: {x_coord}"
    y_coord: float = row["centre_y"]
    assert isinstance(y_coord, float), f"y_coord is not a float: {y_coord}"
    unique_id = create_unique_grain_id(image_name, x_coord, y_coord)
    mapping.append((grain_id, unique_id))

print(f"Created mapping for {len(mapping)} grains.")

# save the mapping to a csv file
pd.DataFrame(mapping, columns=["new_grain_id", "unique_id"]).to_csv(
    path_base_new / "new_grain_id_to_unique_id_mapping.csv", index=False
)

In [ ]:
# Load the two mappings
old_mapping_df = pd.read_csv(path_base_old / "old_grain_id_to_unique_id_mapping.csv")
new_mapping_df = pd.read_csv(path_base_new / "new_grain_id_to_unique_id_mapping.csv")


# find the common rows between the two mappings based on the unique id
common_rows = pd.merge(old_mapping_df, new_mapping_df, on="unique_id", how="inner", suffixes=("_old", "_new"))
common_grains: set[tuple[int, str, int]] = set(
    zip(common_rows["old_grain_id"], common_rows["unique_id"], common_rows["new_grain_id"])
)
# find the not common grains in the old mapping
old_not_common_grains = set(zip(old_mapping_df["old_grain_id"], old_mapping_df["unique_id"])) - set(
    zip(common_rows["old_grain_id"], common_rows["unique_id"])
)
# find the not common grains in the new mapping
new_not_common_grains = set(zip(new_mapping_df["new_grain_id"], new_mapping_df["unique_id"])) - set(
    zip(common_rows["new_grain_id"], common_rows["unique_id"])
)

print(f"Found {len(common_grains)} common grains.")
print(f"Found {len(old_not_common_grains)} old not common grains.")
print(f"Found {len(new_not_common_grains)} new not common grains.")

# check that the dates are okay for each of the not common grains
for old_not_common_grain in old_not_common_grains:
    date_str = old_not_common_grain[1].split("_")[0]
    try:
        date = datetime.strptime(date_str, "%Y%m%d")
    except ValueError as e:
        print(f"Error parsing date from {old_not_common_grain[1]}")
        continue
    assert date < datetime(
        2026, 9, 1
    ), f"Old unique_id {old_not_common_grain[1]} has a date later than September 2026: {date}"
for new_not_common_grain in new_not_common_grains:
    date_str = new_not_common_grain[1].split("_")[0]
    try:
        date = datetime.strptime(date_str, "%Y%m%d")
    except ValueError as e:
        print(f"Error parsing date from {new_not_common_grain[1]}")
        continue
    assert date >= datetime(
        2026, 9, 1
    ), f"New unique_id {new_not_common_grain[1]} has a date earlier than September 2026: {date}"

# save the common grains to a csv file
pd.DataFrame(list(common_grains), columns=["old_grain_id", "unique_id", "new_grain_id"]).to_csv(
    path_base_new / "old_to_new_grain_id_mapping.csv", index=False
)

In [ ]:
# Update the old image tags file to use the new grain ids
old_image_tags_file = path_base_old / "gallery_plots_bk_20260918" / "tagged-output" / "image_tags.csv"
new_image_tags_file = path_base_new / "gallery_plots" / "tagged-output" / "image_tags.csv"
assert old_image_tags_file.exists(), f"Old image tags file does not exist: {old_image_tags_file}"

# Load the old image tags
old_image_tags_df = pd.read_csv(old_image_tags_file)
# rename "Unnamed: 0" to "file_path"
old_image_tags_df.rename(columns={"Unnamed: 0": "file_path"}, inplace=True)
new_image_tags: list[dict] = []
# iterate over each row
for index, row in old_image_tags_df.iterrows():
    # load the row as a dictionary
    row_dict = row.to_dict()
    old_file_path: Path = Path(row_dict["file_path"])
    # get the old grain id
    old_filename: str = row_dict["filename"]
    # load the other columns as a dictionary as well
    old_filename_without_extension: str = old_filename.replace(".png", "")
    # the old grain id is the last number in the filename, separated by a _, after removing the .png
    old_grain_id: int = int(old_filename_without_extension.split("_")[-1])

    # check that the old grain id is in the common grains - note that if I have already filtered this csv file for
    # controls, then this might not even trigger
    if old_grain_id not in [grain_id_data[0] for grain_id_data in common_grains]:
        print(f"Old grain id {old_grain_id} not found in common grains, skipping.")
        continue

    # get the grain_id_data tuple for the old grain id
    matching_grain_id_data: list[tuple[int, str, int]] = [
        grain_id_data for grain_id_data in common_grains if grain_id_data[0] == old_grain_id
    ]
    assert (
        len(matching_grain_id_data) == 1
    ), f"Found {len(matching_grain_id_data)} matching grain id data for old grain id {old_grain_id}, expected 1."
    new_grain_id: int = matching_grain_id_data[0][2]
    # update the filename to use the new grain id
    new_filename: str = old_filename_without_extension.replace(f"_{old_grain_id}", f"_{new_grain_id}") + ".png"
    new_file_path: Path = old_file_path.parent / new_filename
    print(f"Updated old filename {old_filename} to new filename {new_filename}")

    # add the new filename to the row dictionary
    row_dict["filename"] = new_filename
    # update the file path in the row dictionary
    row_dict["file_path"] = str(new_file_path)
    # add the updated row dictionary to the new image tags list
    new_image_tags.append(row_dict)

# save the new image tags to a csv file
new_image_tags_df = pd.DataFrame(new_image_tags)
new_image_tags_df.set_index("file_path", inplace=True)
new_image_tags_df.to_csv(new_image_tags_file, index_label="file_path")

In [ ]:
# create a mapping from new grain ids to uuids using a csv file
defect_grain_statistics_df = pd.read_csv(
    "/Users/sylvi/topo_data/dna_damage_cache/analysis_results/defect_grain_statistics.csv"
)
manual_tagged_df = pd.read_csv(
    Path("/Users/sylvi/topo_data/dna_damage_cache/analysis_results/gallery_plots/tagged-output/image_tags.csv")
)
# make a backup of the manual_tagged_df
manual_tagged_df.to_csv(
    "/Users/sylvi/topo_data/dna_damage_cache/analysis_results/gallery_plots/tagged-output/image_tags_backup.csv", index=False
)
new_defect_grain_statistics_df = pd.read_csv(
    "/Users/sylvi/Downloads/defect_grain_statistics.csv"
)
# change the column name to "file_path"
manual_tagged_df = manual_tagged_df.rename(columns={"Unnamed: 0": "file_path"})
mapping: list[tuple[int, str]] = []
for _, row in defect_grain_statistics_df.iterrows():
    grain_id: int = row["grain_id"]
    # print(f"Processing grain_id: {grain_id}")
    grain_filename = row["image_x"]
    # print(f"Grain filename: {grain_filename}")
    grain_centre_x = row["centre_x"]
    # print(f"Grain centre x: {grain_centre_x}")
    grain_centre_y = row["centre_y"]
    # print(f"Grain centre y: {grain_centre_y}")
    grain_uuid = create_unique_grain_id(grain_filename, grain_centre_x, grain_centre_y)
    # print(f"Grain uuid: {grain_uuid}")
    mapping.append((grain_id, grain_uuid))

    # get the corresponding row in the manual_tagged_df
    old_filename_id = f"{grain_filename}_{grain_id}" + ".png"
    assert old_filename_id in manual_tagged_df["filename"].values, f"Old filename id {old_filename_id} not found in manual_tagged_df"

    row = manual_tagged_df[manual_tagged_df["filename"] == old_filename_id]
    assert len(row) == 1, f"Expected 1 row for old filename id {old_filename_id}, found {len(row)} rows."

    # change the file path
    old_file_path = row["file_path"].values[0]
    old_file_path_parent = Path(old_file_path).parent
    new_file_path = old_file_path_parent / (grain_uuid + ".png")
    # print(f"old file path: {old_file_path}, new file path: {new_file_path}")
    # change the filename
    new_filename = f"{grain_uuid}.png"
    # update the row in the manual_tagged_df
    manual_tagged_df.loc[manual_tagged_df["filename"] == old_filename_id, "filename"] = new_filename
    manual_tagged_df.loc[manual_tagged_df["filename"] == new_filename, "file_path"] = str(new_file_path)


    # check that the new grain id is in the new_defect_grain_statistics_df
    new_row = new_defect_grain_statistics_df[new_defect_grain_statistics_df["grain_id"] == grain_uuid]
    assert len(new_row) == 1, f"Expected 1 row for new grain id {grain_uuid}, found {len(new_row)} rows."

# save the updated manual_tagged_df to a csv file
manual_tagged_df.to_csv(
    "/Users/sylvi/topo_data/dna_damage_cache/analysis_results/gallery_plots/tagged-output/image_tags.csv", index=False
)


print(len(mapping))
# save the mapping to a csv file
pd.DataFrame(mapping, columns=["grain_id", "uuid"]).to_csv(
    "/Users/sylvi/topo_data/dna_damage_cache/analysis_results/grain_id_to_uuid_mapping.csv", index=False
)